# Análogos de la Luna y Marte en México · cuaderno de análisis

**Space Apps 2026** · reto *Identify Earth locations that analog the permanent Moon base locations and Mars*.

Este cuaderno busca en todo México **formas del terreno parecidas a cráteres**, las mide con **el mismo código** que cráteres del **polo sur lunar** y de **Marte**, y exporta los datos que usa la página web (`web/`). También contiene la **validación** del detector y un experimento que la explica.

## Qué hace, en una línea por paso

| Celda | Qué hace | Sale |
|---|---|---|
| 00 | Prepara librerías, Earth Engine y funciones comunes | — |
| 01 | Divide México en cuadros de 10 km y marca los **áridos** (poca vegetación, sin agua, ciudad ni cultivo) | 2,587 de 20,356 cuadros |
| 02 | Baja el **relieve** de esos cuadros y mide pendientes | estadística por cuadro |
| 03 | Baja el relieve de la **Luna** (polo sur) y de **Marte** (Jezero, Gale, Oxia Planum) y lo mide igual | 725 cuadros lunares, 300 marcianos |
| 04 | Compara **pendientes y relieve** de cada cuadro mexicano con cada referencia | distancia 0 = idéntico |
| 05 | Cuenta **depresiones cerradas** por cuadro | — |
| 06 | Puntaje de forma combinado (pendientes + relieve + depresiones) | ranking de cuadros |
| 07 | **Buscador de cráteres**: el mismo detector en México (todo el país), la Luna y Marte | 668 en México, 1,056 en el polo sur |
| 08 | **Validación 1**: ¿encuentra los cráteres mexicanos conocidos? | 9 de 11 |
| 09 | Separa **minas**, mide vegetación y brillo, agrupa en sitios y busca el **gemelo lunar** | tipos y gemelos |
| 10 | **Exporta** los datos de la página web | `web_data.zip` |
| 11 | **Validación 2**: imágenes de relieve y satélite de una muestra al azar | `informe/` |
| 12 | Imágenes de **todas las detecciones en zonas áridas** (la lista que importa) | 47 sitios |
| 13 | **Experimento**: ¿un relieve sin árboles (FABDEM) quita los falsos? | No: solo −15 % |
| 99 | Descarga los zip al equipo | — |

## De dónde salen los datos

Todos son **abiertos**. Los de la Tierra se leen a través de **Google Earth Engine** (hace falta una cuenta y un proyecto de Google Cloud; aquí: `dulcet-cat-514618-d1`). Los de la Luna y Marte se bajan directo de los archivos de NASA y USGS.

| Para qué | Dato | Quién lo produce | Resolución original | Dónde |
|---|---|---|---|---|
| Relieve de México | **Copernicus DEM GLO-30** | ESA / Airbus | 30 m | Earth Engine `COPERNICUS/DEM/GLO30_2024_1` |
| Relieve del polo sur lunar | **LOLA** `ldem_80s_80m` (sonda LRO) | NASA | 80 m | [PDS Geosciences](https://pds-geosciences.wustl.edu/lro/lro-l-lola-3-rdr-v1/lrolol_1xxx/data/lola_gdr/polar/img/) |
| Relieve de Marte | **HRSC + MOLA blend** v2 | ESA (Mars Express) y NASA (MGS), mosaico de USGS | 200 m | [USGS Astrogeology](https://planetarymaps.usgs.gov/mosaic/Mars/HRSC_MOLA_Blend/) |
| Vegetación | **MODIS MOD13Q1** v6.1 (NDVI, 2021–2025) | NASA (LP DAAC) | 250 m | Earth Engine `MODIS/061/MOD13Q1` |
| Cobertura del suelo | **MODIS MCD12Q1** v6.1 (2024) | NASA (LP DAAC) | 500 m | Earth Engine `MODIS/061/MCD12Q1` |
| Brillo de la superficie | **MODIS MCD43A3** (albedo) | NASA (LP DAAC) | 500 m | Earth Engine `MODIS/061/MCD43A3` |
| Fronteras de México | **LSIB** simplificado 2017 | Departamento de Estado de EE. UU. | — | Earth Engine `USDOS/LSIB_SIMPLE/2017` |
| Minas y canteras | **OpenStreetMap** (`landuse=quarry`) | colaboradores de OSM | — | API Overpass |
| Imagen de satélite (solo para revisar) | **Sentinel-2** L2A, mediana 2023–2025 | ESA / Copernicus | 10 m | Earth Engine `COPERNICUS/S2_SR_HARMONIZED` |
| Relieve sin árboles (experimento) | **FABDEM** | Univ. de Bristol (Hawker et al., 2022) | 30 m | Earth Engine `projects/sat-io/open-datasets/FABDEM` |

**Regla clave:** los tres mundos se llevan a **200 m por píxel** y pasan por **el mismo código**. Así la comparación es justa: si un cráter mexicano y uno lunar salen parecidos, no es por usar datos distintos.

## Cómo correrlo

1. Abrir en Google Colab y ejecutar las celdas **en orden** (cada una usa variables de las anteriores).
2. La celda 00 pide autorizar Earth Engine con la cuenta de Google.
3. Tiempo total: unos **30 minutos**. Las más lentas son 01 (~9 min), 07 (~8 min) y 10 (~3 min).
4. Al final, la celda 99 baja `web_data.zip` (para `web/public/data/`) e `informe.zip` (imágenes del PDF de validación).

## Lo que aprendimos (léelo antes de usar los números)

El detector **encuentra 9 de 11 cráteres conocidos**, pero **la mayoría de sus 668 detecciones no son cráteres**: el 71 % cae en sierras con selva del sur, donde hay muchas depresiones reales (dolinas, cañadas cerradas) que a 200 m cumplen las reglas de forma. Un relieve sin árboles (FABDEM, celda 13) no lo arregla. **Lo que funciona es quedarse con las zonas áridas**: ahí están todos los cráteres y minas reales. Detalle con imágenes en `informe/validacion_detector.pdf`.


## 00 · Preparación

Instala `rasterio` si falta, conecta con **Earth Engine** y define lo que usan todas las celdas:

- `CELL = 10 km`: tamaño de los cuadros en que se divide el país.
- `BASE = 200 m`: resolución común para la Tierra, la Luna y Marte.
- `CRS = EPSG:6372`: proyección métrica oficial de México (Lambert), para medir distancias en metros.
- `ee_pixels()`: pide a Earth Engine un rectángulo de píxeles como arreglo de numpy (con reintentos).
- `stats_tile()` y `slope_deg()`: miden pendientes y relieve de un cuadro. **Son las mismas funciones para los tres mundos.**

In [ ]:
# 00 · Preparación: librerías, Earth Engine y funciones comunes
import os, re, sys, time, subprocess
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from pyproj import Transformer
import ee

PROJECT = "dulcet-cat-514618-d1"
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Earth Engine listo:", ee.Number(1).add(1).getInfo() == 2)

try:
    import rasterio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)
    import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_origin

CELL = 10_000            # lado del cuadro, en metros
BASE = 200               # metros por píxel: resolución común para Tierra, Luna y Marte
N = CELL // BASE         # 50 píxeles por lado de cuadro
CRS = "EPSG:6372"        # proyección métrica oficial de México (cónica de Lambert)
PCTS = list(range(5, 100, 5))
OUT = "/content/salida"; os.makedirs(OUT, exist_ok=True)
to_xy = Transformer.from_crs("EPSG:4326", CRS, always_xy=True)
to_ll = Transformer.from_crs(CRS, "EPSG:4326", always_xy=True)

def ee_pixels(img, x_left, y_top, width, height, scale, crs=CRS, tries=4):
    """Pide a Earth Engine un rectángulo de píxeles como arreglo de numpy."""
    grid = {"dimensions": {"width": int(width), "height": int(height)},
            "affineTransform": {"scaleX": scale, "shearX": 0, "translateX": float(x_left),
                                "shearY": 0, "scaleY": -scale, "translateY": float(y_top)},
            "crsCode": crs}
    err = ""
    for k in range(tries):
        try:
            return ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY", "grid": grid})
        except Exception as e:
            err = str(e)[:200]; time.sleep(2 * (k + 1))
    raise RuntimeError(err)

def to_grid(img, native_proj, reducer=None):
    """Resume una imagen fina dentro de cada píxel de salida (promedio por defecto)."""
    return img.setDefaultProjection(native_proj).reduceResolution(reducer or ee.Reducer.mean(), maxPixels=4096)

def stats_tile(s, e):
    """Medidas de terreno de un cuadro: s = pendientes en grados, e = elevaciones en metros."""
    s, e = s.ravel(), e.ravel()
    d = {f"slope_p{p}": v for p, v in zip(PCTS, np.percentile(s, PCTS))}
    d.update({"slope_mean": s.mean(), "slope_stdDev": s.std()})
    d.update({f"elev_p{p}": v for p, v in zip((5, 50, 95), np.percentile(e, (5, 50, 95)))})
    d["elev_stdDev"] = e.std(); d["relieve"] = d["elev_p95"] - d["elev_p5"]
    return d

def slope_deg(dem):
    """Pendiente en grados por diferencias centrales, sobre un mapa de elevación a BASE metros por píxel."""
    gy, gx = np.gradient(dem.astype("float64"), BASE)
    return np.degrees(np.arctan(np.hypot(gx, gy)))

print("Listo. Salidas en", OUT)

## 01 · Malla de México y filtro de zonas áridas

**Datos:** MODIS MOD13Q1 (vegetación, NDVI 2021–2025), MODIS MCD12Q1 (cobertura del suelo 2024), fronteras LSIB.

Divide México en cuadros de 10 km y para cada uno calcula: NDVI medio y **NDVI en época verde** (percentil 90), y qué fracción es suelo desnudo, ciudad, agua, cultivo, bosque, matorral o pastizal.

**Filtro de zona árida** (un cuadro pasa si cumple todo):
- NDVI en época verde **< 0.30** (poca vegetación incluso en temporada de lluvias),
- agua < 10 %, ciudad < 5 %, cultivo < 10 %.

El umbral 0.30 se eligió para que entren los análogos conocidos (**El Pinacate**: 0.22, **Cuatro Ciénegas**: 0.23). La tabla final es una **prueba de control**: la Selva Lacandona y el valle de riego de Mexicali deben quedar fuera.

**Resultado:** 2,587 de 20,356 cuadros (13 %) son áridos.

In [ ]:
# 01 · Malla de México (cuadros de 10 km) y filtro de zonas áridas
# Datos: MODIS MOD13Q1 (vegetación) y MCD12Q1 (cobertura del suelo), NASA; límites de país LSIB.
mexico = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq("country_na", "Mexico")).geometry()
b = np.array(mexico.bounds(100, ee.Projection(CRS)).coordinates().getInfo()[0])
x0, y1 = np.floor(b[:, 0].min() / CELL) * CELL, np.ceil(b[:, 1].max() / CELL) * CELL
nx = int(np.ceil((b[:, 0].max() - x0) / CELL)); ny = int(np.ceil((y1 - b[:, 1].min()) / CELL))

def fetch_grid(img, need=None, block=24, workers=8):
    """Trae una imagen ya resumida a la malla de 10 km, por bloques. `need` limita a los bloques con cuadros de interés."""
    jobs = [(r, c) for r in range(0, ny, block) for c in range(0, nx, block)
            if need is None or need[r:r + block, c:c + block].any()]
    def one(rc):
        r, c = rc
        return r, c, ee_pixels(img, x0 + c * CELL, y1 - r * CELL, min(block, nx - c), min(block, ny - r), CELL)
    with ThreadPoolExecutor(workers) as ex:
        parts = list(ex.map(one, jobs))
    names = parts[0][2].dtype.names
    out = {n: np.full((ny, nx), np.nan, "float32") for n in names}
    for r, c, a in parts:
        for n in names:
            out[n][r:r + a.shape[0], c:c + a.shape[1]] = a[n]
    return out

t0 = time.time()
ndvi_col = ee.ImageCollection("MODIS/061/MOD13Q1").filterDate("2021-01-01", "2026-01-01").select("NDVI")
p_ndvi = ndvi_col.first().projection()
lc = ee.ImageCollection("MODIS/061/MCD12Q1").filterDate("2024-01-01", "2025-01-01").first().select("LC_Type1")
p_lc = lc.projection()
frac = lambda codes, name: to_grid(lc.remap(codes, [1] * len(codes), 0), p_lc).rename(name)
bands = [to_grid(ndvi_col.mean().multiply(1e-4), p_ndvi).rename("ndvi_mean"),
         to_grid(ndvi_col.reduce(ee.Reducer.percentile([90])).multiply(1e-4), p_ndvi).rename("ndvi_p90"),
         frac([16], "f_desnudo"), frac([13], "f_urbano"), frac([17], "f_agua"), frac([12, 14], "f_cultivo"),
         frac([1, 2, 3, 4, 5, 8, 9], "f_bosque"), frac([6, 7], "f_matorral"), frac([10], "f_pastizal"),
         ee.Image(1).clip(mexico).unmask(0).rename("en_mexico")]   # 1 si el centro del cuadro cae dentro de México
g = fetch_grid(ee.Image.cat(bands).toFloat().unmask(-1))

rows, cols = np.mgrid[0:ny, 0:nx]
df = pd.DataFrame({n: g[n].ravel() for n in g})
df["row"], df["col"] = rows.ravel(), cols.ravel()
df["x"] = x0 + (df["col"] + 0.5) * CELL; df["y"] = y1 - (df["row"] + 0.5) * CELL
df["lon"], df["lat"] = to_ll.transform(df["x"].values, df["y"].values)
mx = df[df["en_mexico"] >= 0.5].copy()

# Filtro: poca vegetación incluso en época verde, y casi sin agua, ciudad ni cultivo.
# Umbral 0.30: El Pinacate (0.22) y Cuatro Ciénegas (0.23) quedan dentro con margen.
NDVI_MAX = 0.30
ok = (mx.f_agua < 0.10) & (mx.f_urbano < 0.05) & (mx.f_cultivo < 0.10) & (mx.ndvi_mean >= 0) & (mx.ndvi_p90 < NDVI_MAX)
cand = mx[ok].reset_index(drop=True)
cand.insert(0, "cell_id", "MX_" + cand.row.astype(str).str.zfill(3) + "_" + cand.col.astype(str).str.zfill(3))
cand.to_csv(f"{OUT}/candidatos_mexico_10km.csv", index=False)
print(f"Malla {nx} x {ny}. Cuadros en México: {len(mx)}. Candidatos áridos: {len(cand)} ({len(cand)/len(mx):.0%}). Tiempo {time.time()-t0:.0f} s")

def celda(lat, lon):
    x, y = to_xy.transform(lon, lat)
    return int((y1 - y) // CELL), int((x - x0) // CELL)
CONTROL = {"El Pinacate": (31.77, -113.49), "Cuatro Ciénegas": (26.87, -102.12), "Samalayuca": (31.30, -106.50),
           "Desierto de Vizcaíno": (27.50, -113.80), "Selva Lacandona": (16.50, -91.00), "Valle de Mexicali (riego)": (32.45, -115.20)}
ver = ["ndvi_p90", "f_desnudo", "f_matorral", "f_cultivo", "f_urbano", "f_agua"]
print(pd.DataFrame({k: df[(df.row == celda(*v)[0]) & (df.col == celda(*v)[1])].iloc[0][ver] for k, v in CONTROL.items()}).T.astype(float).round(2))

## 02 · Relieve de los cuadros áridos

**Dato:** Copernicus DEM GLO-30 (ESA). Se pide ya **promediado a 200 m** y se mide con el mismo código que la Luna y Marte.

Por cada cuadro árido: distribución de pendientes (percentiles 5 a 95), pendiente media y desviación, y **relieve** (diferencia entre la elevación alta y la baja). También guarda el relieve de cada cuadro (`parches_mexico.npz`).

**Ojo:** GLO-30 es un modelo de **superficie** (incluye copas de árboles y edificios), hecho con radar.

In [ ]:
# 02 · Terreno de los candidatos (Tierra)
# Datos: Copernicus DEM GLO-30 (ESA). Se baja la elevación ya promediada a 200 m y se mide con el MISMO código que la Luna y Marte.
glo = ee.ImageCollection("COPERNICUS/DEM/GLO30_2024_1").select("DEM")
dem_ee = (glo.mosaic().setDefaultProjection(glo.first().projection())
          .reduceResolution(ee.Reducer.mean(), maxPixels=256).unmask(0).toFloat().rename("dem"))

BLOCK = 16                                   # cuadros por lado de cada descarga (160 km = 800 píxeles)
need = np.zeros((ny, nx), bool); need[cand.row.values, cand.col.values] = True
jobs = [(r, c) for r in range(0, ny, BLOCK) for c in range(0, nx, BLOCK) if need[r:r + BLOCK, c:c + BLOCK].any()]

def bloque_terreno(rc):
    r, c = rc
    h, w = min(BLOCK, ny - r), min(BLOCK, nx - c)
    # un píxel extra por lado para que la pendiente en el borde use vecinos reales
    a = ee_pixels(dem_ee, x0 + c * CELL - BASE, y1 - r * CELL + BASE, w * N + 2, h * N + 2, BASE)["dem"]
    s, e = slope_deg(a)[1:-1, 1:-1], a[1:-1, 1:-1]
    out = []
    for i in range(h):
        for j in range(w):
            if need[r + i, c + j]:
                et = e[i * N:(i + 1) * N, j * N:(j + 1) * N]
                st = stats_tile(s[i * N:(i + 1) * N, j * N:(j + 1) * N], et)
                st.update(row=r + i, col=c + j)
                out.append((st, et.astype("float32")))
    return out

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    res = [x for part in ex.map(bloque_terreno, jobs) for x in part]
terr = pd.DataFrame([st for st, _ in res])
cand = cand.drop(columns=[c for c in terr.columns if c in cand.columns and c not in ("row", "col")]).merge(terr, on=["row", "col"], how="left")
parches_mx = {f"MX_{st['row']:03d}_{st['col']:03d}": p for st, p in res}      # relieve de cada cuadro, para el juego y la red
np.savez_compressed(f"{OUT}/parches_mexico.npz", **parches_mx)
cand.to_csv(f"{OUT}/candidatos_mexico_10km.csv", index=False)
print(f"Bloques: {len(jobs)}. Cuadros medidos: {cand.slope_mean.notna().sum()} de {len(cand)}. Tiempo {time.time()-t0:.0f} s")
print(cand[["slope_p50", "slope_p90", "slope_mean", "slope_stdDev", "relieve"]].describe().T[["mean", "min", "50%", "max"]].round(2))
for k in ("El Pinacate", "Cuatro Ciénegas", "Samalayuca", "Desierto de Vizcaíno"):
    r_, c_ = celda(*CONTROL[k]); m = cand[(cand.row == r_) & (cand.col == c_)]
    print(k, "->", m[["slope_p50", "slope_p90", "slope_stdDev", "relieve"]].round(2).to_dict("records")[0] if len(m) else "no es candidato")

## 03 · Referencias: Luna y Marte

**Luna:** LOLA (sonda LRO de NASA), producto polar `ldem_80s_80m` de 80 m, bajado directo de PDS y **promediado a 200 m**. Se divide en cuadros de 10 km; se separa el **polo sur (85–90° S)**, donde estaría la base Artemis, del anillo de 80–85° S.

**Marte:** mosaico HRSC + MOLA de 200 m (USGS). Se recortan 100 × 100 km alrededor de tres sitios: **cráter Jezero** (rover Perseverance), **cráter Gale** (rover Curiosity) y **Oxia Planum** (rover Rosalind Franklin).

La tabla compara la mediana de pendientes y relieve de cada grupo con los cuadros áridos de México. El polo sur lunar es mucho más empinado (pendiente típica de 8°) que casi todo México (1°).

In [ ]:
# 03 · Referencias de la Luna y Marte, a los mismos 200 m por píxel y en cuadros de 10 km
# Luna: LOLA (LRO, NASA), producto polar ldem_80s_80m. Marte: mosaico HRSC+MOLA de 200 m (ESA/NASA vía USGS).
def terrain_stats(dem, valid=None):
    """Una fila por cuadro de 10 km de un mapa de elevación a 200 m. Devuelve también los recortes de relieve."""
    s = slope_deg(dem); rows, patches = [], []
    for i in range(0, dem.shape[0] - N + 1, N):
        for j in range(0, dem.shape[1] - N + 1, N):
            if valid is not None and not valid[i:i + N, j:j + N].all():
                continue
            d = stats_tile(s[i:i + N, j:j + N], dem[i:i + N, j:j + N]); d.update(ti=i // N, tj=j // N)
            rows.append(d); patches.append(dem[i:i + N, j:j + N].astype("float32"))
    return pd.DataFrame(rows), patches

# ---------- Luna ----------
os.makedirs("/content/luna", exist_ok=True)
LOLA = "https://pds-geosciences.wustl.edu/lro/lro-l-lola-3-rdr-v1/lrolol_1xxx/data/lola_gdr/polar/img/"
for f in ("ldem_80s_80m.lbl", "ldem_80s_80m.img"):
    if not os.path.exists(f"/content/luna/{f}"):
        subprocess.run(["wget", "-q", "-O", f"/content/luna/{f}", LOLA + f], check=True)
lbl = open("/content/luna/ldem_80s_80m.lbl", errors="replace").read()
val = lambda k: re.search(rf"^\s*{k}\s*=\s*([-\d.]+)", lbl, re.M).group(1)
LINES, SCALE, MAPSC = int(val("LINES")), float(val("SCALING_FACTOR")), float(val("MAP_SCALE"))
assert (LINES, SCALE, MAPSC) == (7600, 0.5, 80.0), "la etiqueta PDS no es la esperada"
R_MOON = 1737400.0
crs_moon = f"+proj=stere +lat_0=-90 +lon_0=0 +k=1 +x_0=0 +y_0=0 +R={R_MOON} +units=m +no_defs"
src = np.fromfile("/content/luna/ldem_80s_80m.img", dtype="<i2").reshape(LINES, LINES).astype("float32") * SCALE   # metros
half = LINES * MAPSC / 2; n200 = int(LINES * MAPSC / BASE)
moon200 = np.zeros((n200, n200), "float32")
reproject(src, moon200, src_transform=from_origin(-half, half, MAPSC, MAPSC), src_crs=crs_moon,
          dst_transform=from_origin(-half, half, BASE, BASE), dst_crs=crs_moon, resampling=Resampling.average)
luna, p_luna = terrain_stats(moon200)
xm = -half + (luna.tj + 0.5) * CELL; ym = half - (luna.ti + 0.5) * CELL
luna["lon"], luna["lat"] = Transformer.from_crs(crs_moon, f"+proj=longlat +R={R_MOON} +no_defs", always_xy=True).transform(xm.values, ym.values)
luna["cuerpo"] = "Luna"; luna["sitio"] = np.where(luna.lat <= -85, "Polo sur (85-90°S)", "Anillo 80-85°S")
keep = (luna.lat <= -80.5).values
luna, p_luna = luna[keep].reset_index(drop=True), [p for p, k in zip(p_luna, keep) if k]

# ---------- Marte ----------
R_MARS = 3396190.0
MARS_URL = "/vsicurl/https://planetarymaps.usgs.gov/mosaic/Mars/HRSC_MOLA_Blend/Mars_HRSC_MOLA_BlendDEM_Global_200mp_v2.tif"
SITIOS_MARTE = {"Cráter Jezero": (18.44, 77.45), "Cráter Gale": (-5.40, 137.80), "Oxia Planum": (18.20, -24.30)}  # lat, lon este; por verificar contra el nomenclátor IAU
marte, p_marte = [], []
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif"):
    with rasterio.open(MARS_URL) as ds:
        for name, (la, lo) in SITIOS_MARTE.items():
            crs_loc = f"+proj=eqc +lat_ts={la} +lat_0=0 +lon_0={lo} +x_0=0 +y_0=0 +R={R_MARS} +units=m +no_defs"
            ytop = R_MARS * np.radians(la) + 50_000
            with WarpedVRT(ds, crs=crs_loc, transform=from_origin(-50_000, ytop, BASE, BASE), width=500, height=500, resampling=Resampling.bilinear) as vrt:
                a = vrt.read(1).astype("float32")
            okm = np.ones_like(a, bool) if ds.nodata is None else (a != ds.nodata)
            t, pp = terrain_stats(a, okm)
            xs = -50_000 + (t.tj + 0.5) * CELL; ys = ytop - (t.ti + 0.5) * CELL
            t["lon"], t["lat"] = Transformer.from_crs(crs_loc, f"+proj=longlat +R={R_MARS} +no_defs", always_xy=True).transform(xs.values, ys.values)
            t["cuerpo"], t["sitio"] = "Marte", name
            print(f"  {name}: {len(t)} cuadros, elevación {a[okm].min():.0f} a {a[okm].max():.0f} m")
            marte.append(t); p_marte += pp

ref = pd.concat([luna] + marte, ignore_index=True)
ref.insert(0, "cell_id", ref.cuerpo.str[:2].str.upper() + "_" + ref.index.astype(str).str.zfill(4))
parches_ref = dict(zip(ref.cell_id, p_luna + p_marte))
np.savez_compressed(f"{OUT}/parches_referencias.npz", **parches_ref)
ref.to_csv(f"{OUT}/referencias_luna_marte_10km.csv", index=False)
cmp_cols = ["slope_p50", "slope_p90", "slope_mean", "slope_stdDev", "relieve"]
tabla = ref.groupby(["cuerpo", "sitio"])[cmp_cols].median()
tabla["cuadros"] = ref.groupby(["cuerpo", "sitio"]).size()
tabla.loc[("Tierra", "Candidatos México"), :] = list(cand[cmp_cols].median().values) + [len(cand)]
print("\nMediana por grupo (pendientes en grados, relieve en metros):"); print(tabla.round(2).to_string())

## 04 · Comparación de terreno

Para cada cuadro mexicano y cada referencia calcula **qué tan parecido es el terreno**:

- **Pendientes:** diferencia media entre los 19 percentiles de pendiente (equivale a la distancia de Wasserstein entre las dos distribuciones).
- **Relieve:** diferencia en logaritmo.

Cada distancia se divide entre lo que varía la propia referencia (su percentil 90). Así, **d ≤ 1 significa "tan parecido a la referencia como la referencia lo es a sí misma"**.

**Prueba de humo:** El Pinacate y Cuatro Ciénegas deben quedar arriba frente a la Luna; las dunas de Samalayuca y las llanuras de Vizcaíno, no (y Vizcaíno sí debe parecerse a Oxia Planum, que también es llano). Solo mide **forma del terreno**: nada de composición, temperatura ni luz.

In [ ]:
# 04 · Comparación de terreno y prueba de humo
# Distancia de cada cuadro mexicano al perfil típico (mediana) de cada referencia:
#   pendientes = diferencia media entre los 19 percentiles (distancia de Wasserstein), relieve = diferencia en logaritmo.
# Cada una se divide entre el percentil 90 de esa misma distancia dentro de la referencia:
#   d <= 1 significa "tan parecido a la referencia como la referencia a sí misma". Solo mide forma del terreno.
Q = [f"slope_p{p}" for p in PCTS]

def distancias(tabla, grupo):
    prof = grupo[Q].median().values
    rel = np.log(grupo.relieve.clip(lower=1)).median()
    d_s = lambda t: np.abs(t[Q].values - prof).mean(axis=1)
    d_r = lambda t: np.abs(np.log(t.relieve.clip(lower=1).values) - rel)
    s90, r90 = np.percentile(d_s(grupo), 90), np.percentile(d_r(grupo), 90)
    zs, zr = d_s(tabla) / s90, d_r(tabla) / r90
    return zs, zr, np.maximum(zs, zr)

GRUPOS = {"luna": ("Luna · polo sur", ref[ref.sitio == "Polo sur (85-90°S)"]), "jezero": ("Marte · Jezero", ref[ref.sitio == "Cráter Jezero"]),
          "gale": ("Marte · Gale", ref[ref.sitio == "Cráter Gale"]), "oxia": ("Marte · Oxia Planum", ref[ref.sitio == "Oxia Planum"])}
PRUEBA = ["El Pinacate", "Cuatro Ciénegas", "Samalayuca", "Desierto de Vizcaíno"]
resumen = []
for k, (nombre, g) in GRUPOS.items():
    cand[f"zs_{k}"], cand[f"zr_{k}"], cand[f"d_{k}"] = distancias(cand, g)
    cand[f"rank_{k}"] = cand[f"d_{k}"].rank(method="min").astype(int)
    fila = {"referencia": nombre, "cuadros ref": len(g), "mexicanos con d<=1": int((cand[f"d_{k}"] <= 1).sum())}
    for lugar in PRUEBA:
        x, y = to_xy.transform(CONTROL[lugar][1], CONTROL[lugar][0])
        cerca = cand[np.hypot(cand.x - x, cand.y - y) <= 25_000]
        fila[lugar] = f"{cerca[f'd_{k}'].min():.2f} (#{int(cerca[f'rank_{k}'].min())})" if len(cerca) else "sin cuadros"
    resumen.append(fila)
print(f"Mejor cuadro a menos de 25 km de cada lugar de control: distancia y lugar entre {len(cand)} candidatos")
print(pd.DataFrame(resumen).to_string(index=False))
print("\nLos 8 cuadros mexicanos más parecidos al polo sur lunar (solo terreno):")
print(cand.nsmallest(8, "d_luna")[["cell_id", "lat", "lon", "slope_p50", "slope_p90", "relieve", "d_luna"]].round(2).to_string(index=False))
cand.to_csv(f"{OUT}/candidatos_mexico_10km.csv", index=False)

## 05 · Depresiones cerradas por cuadro

"Rellena" el relieve como si lloviera (reconstrucción morfológica): lo que quedaría bajo el agua es una **depresión cerrada**. Por cuadro cuenta cuántas hay, su profundidad máxima, qué fracción del área ocupan y qué tan redondas son.

Límite: solo ve depresiones que caben en el cuadro de 10 km y miden al menos ~400 m.

In [ ]:
# 05 · Depresiones cerradas (cráteres, maares, hoyas) en los tres cuerpos
# Se "rellena" cada cuadro como si lloviera y se mide lo que quedaría bajo el agua: es la forma más directa de ver cráteres.
# Límite: solo detecta depresiones que caben dentro del cuadro de 10 km y miden al menos ~400 m.
from skimage.morphology import reconstruction
from scipy import ndimage as ndi

PROF_MIN = 5.0      # metros: profundidad mínima para contar como depresión (por encima del ruido a 200 m)
AREA_MIN = 4        # píxeles: al menos 0.16 km2

def depresiones(dem):
    dem = dem.astype("float64")
    seed = dem.copy(); seed[1:-1, 1:-1] = dem.max()
    prof = reconstruction(seed, dem, method="erosion") - dem           # profundidad del relleno en cada píxel
    lab, n = ndi.label(prof > PROF_MIN)
    if n == 0:
        return {"dep_frac": 0.0, "dep_n": 0, "dep_prof_max": 0.0, "dep_circ": 0.0}
    idx = np.arange(1, n + 1)
    area = ndi.sum(np.ones_like(prof), lab, idx)
    ok = area >= AREA_MIN
    if not ok.any():
        return {"dep_frac": 0.0, "dep_n": 0, "dep_prof_max": 0.0, "dep_circ": 0.0}
    # circularidad: área frente a la del círculo que encierra la caja de la depresión (1 = círculo perfecto)
    circ = []
    for sl, a, o in zip(ndi.find_objects(lab), area, ok):
        if o:
            h, w = sl[0].stop - sl[0].start, sl[1].stop - sl[1].start
            circ.append(min(1.0, a / (np.pi * (max(h, w) / 2) ** 2)))
    pmax = ndi.maximum(prof, lab, idx)[ok]
    return {"dep_frac": float(area[ok].sum() / prof.size), "dep_n": int(ok.sum()),
            "dep_prof_max": float(pmax.max()), "dep_circ": float(np.mean(circ))}

t0 = time.time()
dep_mx = pd.DataFrame([dict(cell_id=k, **depresiones(p)) for k, p in parches_mx.items()])
dep_ref = pd.DataFrame([dict(cell_id=k, **depresiones(p)) for k, p in parches_ref.items()])
DEP = ["dep_frac", "dep_n", "dep_prof_max", "dep_circ"]
cand = cand.drop(columns=[c for c in DEP if c in cand.columns]).merge(dep_mx, on="cell_id", how="left")
ref = ref.drop(columns=[c for c in DEP if c in ref.columns]).merge(dep_ref, on="cell_id", how="left")
t = ref.groupby(["cuerpo", "sitio"])[DEP].median()
t["con depresiones %"] = (ref.assign(tiene=ref.dep_n > 0).groupby(["cuerpo", "sitio"]).tiene.mean() * 100).round(0)
t.loc[("Tierra", "Candidatos México"), :] = list(cand[DEP].median().values) + [round((cand.dep_n > 0).mean() * 100)]
print(f"Tiempo {time.time()-t0:.0f} s. Mediana por grupo:"); print(t.round(3).to_string())
for k in PRUEBA:
    x, y = to_xy.transform(CONTROL[k][1], CONTROL[k][0])
    cerca = cand[np.hypot(cand.x - x, cand.y - y) <= 25_000]
    print(f"{k}: de {len(cerca)} cuadros cercanos, {int((cerca.dep_n > 0).sum())} con depresiones; más profunda {cerca.dep_prof_max.max():.0f} m; máx. por cuadro {int(cerca.dep_n.max())}")
cand.to_csv(f"{OUT}/candidatos_mexico_10km.csv", index=False); ref.to_csv(f"{OUT}/referencias_luna_marte_10km.csv", index=False)

## 06 · Puntaje de forma combinado

Junta tres criterios: **pendientes**, **relieve** y **depresiones**. El puntaje final es **el peor de los tres**: un cuadro solo sale bien si se parece en todo, no en promedio.

Produce el ranking de cuadros mexicanos más parecidos al polo sur lunar y a cada sitio marciano. La columna `ref. consigo misma` dice qué porcentaje de la propia referencia pasaría el umbral (para calibrar qué es "parecido").

In [ ]:
# 06 · Puntaje combinado de forma: pendientes + relieve + depresiones
# La distancia de depresiones compara número, profundidad máxima (en log) y fracción de área, cada una en unidades
# de la dispersión interna de la referencia (percentil 90). El puntaje final es el peor de los tres criterios:
# un cuadro solo sale bien si se parece en todo, no en promedio.
SITIO_REF = {"luna": ("Luna · polo sur", "Polo sur (85-90°S)"), "jezero": ("Marte · Jezero", "Cráter Jezero"),
             "gale": ("Marte · Gale", "Cráter Gale"), "oxia": ("Marte · Oxia Planum", "Oxia Planum")}
GRUPOS = {k: (n, ref[ref.sitio == s]) for k, (n, s) in SITIO_REF.items()}   # se rehace: `ref` ya trae las columnas de depresiones

def z_dep(tabla, grupo):
    feats = [lambda t: t["dep_n"].values.astype(float), lambda t: np.log1p(t["dep_prof_max"].values), lambda t: t["dep_frac"].values]
    zs = []
    for f in feats:
        med = np.median(f(grupo)); s90 = max(np.percentile(np.abs(f(grupo) - med), 90), 1e-9)
        zs.append(np.abs(f(tabla) - med) / s90)
    return np.mean(zs, axis=0)

resumen = []
for k, (nombre, g) in GRUPOS.items():
    cand[f"zd_{k}"] = z_dep(cand, g)
    cand[f"f_{k}"] = cand[[f"zs_{k}", f"zr_{k}", f"zd_{k}"]].max(axis=1)
    cand[f"frank_{k}"] = cand[f"f_{k}"].rank(method="min").astype(int)
    propio = np.maximum(distancias(g, g)[2], z_dep(g, g))
    fila = {"referencia": nombre, "mexicanos con f<=1": int((cand[f"f_{k}"] <= 1).sum()), "ref. consigo misma f<=1 %": round((propio <= 1).mean() * 100)}
    for lugar in PRUEBA:
        x, y = to_xy.transform(CONTROL[lugar][1], CONTROL[lugar][0])
        cerca = cand[np.hypot(cand.x - x, cand.y - y) <= 25_000]
        fila[lugar] = f"{cerca[f'f_{k}'].min():.2f} (#{int(cerca[f'frank_{k}'].min())})"
    resumen.append(fila)
print(f"Forma completa. Mejor cuadro a menos de 25 km de cada lugar de control: puntaje y lugar entre {len(cand)}")
print(pd.DataFrame(resumen).to_string(index=False))

xp, yp = to_xy.transform(CONTROL["El Pinacate"][1], CONTROL["El Pinacate"][0])
cand["km_pinacate"] = (np.hypot(cand.x - xp, cand.y - yp) / 1000).round(0)
cols = ["cell_id", "lat", "lon", "slope_p50", "slope_p90", "relieve", "dep_n", "dep_prof_max", "ndvi_p90", "f_desnudo", "f_luna", "km_pinacate"]
print("\nLos 20 cuadros mexicanos más parecidos al polo sur lunar (forma completa):")
print(cand.nsmallest(20, "f_luna")[cols].round(2).to_string(index=False))
cand.to_csv(f"{OUT}/candidatos_mexico_10km.csv", index=False)

## 07 · Buscador de cráteres (el corazón del análisis)

Recorre el relieve en **ventanas de 12 km** con traslape y en cada una "rellena" el terreno para encontrar hoyos cerrados. Por cada hoyo mide: posición, diámetro, profundidad, **redondez**, **altura del borde** frente al entorno y proporción **profundidad/diámetro**.

**Filtro "tipo cráter":** diámetro de 0.6 a 10 km, profundidad ≥ 20 m y redondez ≥ 0.6.

Corre **igual** en México (**todo el país**, bloques de 160 km con margen de 12 km para no cortar cráteres), en el polo sur lunar y en los tres sitios de Marte.

**Resultado:** 5,208 hoyos cerrados en México, de los que **668 pasan el filtro**; 1,056 en el polo sur lunar; 284 en Marte.

**Importante:** aquí se busca en todo el país; el filtro de zonas áridas de la celda 01 **no se aplica**. Eso explica la mayoría de los falsos positivos (ver celdas 11 a 13).

In [ ]:
# 07 · Buscador de cráteres: mismo detector en México (todo el país), la Luna y Marte
# Recorre el relieve en ventanas de 12 km con traslape, "rellena" cada ventana y se queda con las hoyas cerradas.
# Límite: a 200 m por píxel solo ve cráteres de unos 0.6 a 8 km de diámetro.
from skimage.morphology import reconstruction
from scipy import ndimage as ndi

T, S = 60, 30                 # ventana de 12 km, paso de 6 km
PROF_BASE, PROF_C = 5.0, 20.0 # metros: contorno de la hoya y profundidad mínima para contarla
AREA_C = 7                    # píxeles (0.28 km2, diámetro ~0.6 km)

def detectar(dem):
    """Devuelve una fila por depresión cerrada: posición en píxeles, tamaño, profundidad, forma y borde."""
    H, W = dem.shape; out = []
    rng = lambda L: sorted(set(list(range(0, L - T + 1, S)) + [L - T]))
    for i0 in rng(H):
        for j0 in rng(W):
            t = dem[i0:i0 + T, j0:j0 + T].astype("float64")
            if t.max() - t.min() < PROF_C:
                continue
            seed = t.copy(); seed[1:-1, 1:-1] = t.max()
            prof = reconstruction(seed, t, method="erosion") - t
            if prof.max() < PROF_C:
                continue
            lab, n = ndi.label(prof > PROF_BASE)
            for k, sl in enumerate(ndi.find_objects(lab), 1):
                m = lab[sl] == k; area = int(m.sum())
                if area < AREA_C:
                    continue
                pmax = float(prof[sl][m].max())
                if pmax < PROF_C:
                    continue
                h, w = m.shape
                ci, cj = ndi.center_of_mass(m)
                mk = lab == k
                d3 = ndi.binary_dilation(mk, iterations=3)
                r1 = ndi.binary_dilation(mk, iterations=2) & ~mk
                r2 = ndi.binary_dilation(mk, iterations=6) & ~d3
                borde = float(t[r1].mean() - t[r2].mean()) if r2.any() and r1.any() else np.nan
                out.append({"i": i0 + sl[0].start + ci, "j": j0 + sl[1].start + cj, "area_px": area, "prof_m": pmax,
                            "relleno": min(1.0, area / (np.pi / 4 * h * w)), "aspecto": min(h, w) / max(h, w), "borde_m": borde,
                            "elev_m": float(t[mk].min())})
    d = pd.DataFrame(out)
    if d.empty:
        return d
    d["diam_km"] = 2 * np.sqrt(d.area_px * (BASE / 1000) ** 2 / np.pi)
    d["circ"] = d.relleno * d.aspecto
    d["prof_diam"] = d.prof_m / (d.diam_km * 1000)
    # la misma hoya aparece en varias ventanas: se conserva la detección más completa
    d = d.sort_values(["area_px", "prof_m"], ascending=False).reset_index(drop=True)
    keep, ki, kj, kr = [], [], [], []
    for r in d.itertuples():
        rad = r.diam_km * 1000 / BASE / 2 + 1
        if ki and (np.hypot(np.array(ki) - r.i, np.array(kj) - r.j) < np.maximum(np.array(kr), rad)).any():
            continue
        keep.append(r.Index); ki.append(r.i); kj.append(r.j); kr.append(rad)
    return d.loc[keep].reset_index(drop=True)

# ---------- México: todo el país ----------
M = 60                                              # margen de 12 km por lado para no cortar cráteres entre bloques
en_mx = np.zeros((ny, nx), bool); en_mx[mx.row.values, mx.col.values] = True
jobs = [(r, c) for r in range(0, ny, BLOCK) for c in range(0, nx, BLOCK) if en_mx[r:r + BLOCK, c:c + BLOCK].any()]
dem_bloques = {}
def bloque_crateres(rc):
    r, c = rc
    h, w = min(BLOCK, ny - r), min(BLOCK, nx - c)
    xl, yt = x0 + c * CELL - M * BASE, y1 - r * CELL + M * BASE
    a = ee_pixels(dem_ee, xl, yt, w * N + 2 * M, h * N + 2 * M, BASE)["dem"]
    dem_bloques[rc] = (a, xl, yt)
    d = detectar(a)
    if d.empty:
        return d
    d = d[(d.i >= M) & (d.i < M + h * N) & (d.j >= M) & (d.j < M + w * N)].copy()   # solo el núcleo del bloque
    d["x"], d["y"] = xl + (d.j + 0.5) * BASE, yt - (d.i + 0.5) * BASE
    d["br"], d["bc"] = r, c
    return d
t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    hoyas_mx = pd.concat(list(ex.map(bloque_crateres, jobs)), ignore_index=True)
hoyas_mx["lon"], hoyas_mx["lat"] = to_ll.transform(hoyas_mx.x.values, hoyas_mx.y.values)
rr, cc = ((y1 - hoyas_mx.y) // CELL).astype(int), ((hoyas_mx.x - x0) // CELL).astype(int)
hoyas_mx = hoyas_mx[en_mx[rr.clip(0, ny - 1), cc.clip(0, nx - 1)]].reset_index(drop=True)
print(f"México: {len(jobs)} bloques, {len(hoyas_mx)} depresiones cerradas de al menos {PROF_C:.0f} m. Tiempo {time.time()-t0:.0f} s")

# ---------- Luna y Marte ----------
hl = detectar(moon200)
hl["lon"], hl["lat"] = Transformer.from_crs(crs_moon, f"+proj=longlat +R={R_MOON} +no_defs", always_xy=True).transform(
    (-half + (hl.j + 0.5) * BASE).values, (half - (hl.i + 0.5) * BASE).values)
hl = hl[hl.lat <= -80.5].copy(); hl["cuerpo"] = "Luna"; hl["sitio"] = np.where(hl.lat <= -85, "Polo sur (85-90°S)", "Anillo 80-85°S")
hm = []; dem_marte = {}
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif"):
    with rasterio.open(MARS_URL) as ds:
        for name, (la, lo) in SITIOS_MARTE.items():
            crs_loc = f"+proj=eqc +lat_ts={la} +lat_0=0 +lon_0={lo} +x_0=0 +y_0=0 +R={R_MARS} +units=m +no_defs"
            ytop = R_MARS * np.radians(la) + 50_000
            with WarpedVRT(ds, crs=crs_loc, transform=from_origin(-50_000, ytop, BASE, BASE), width=500, height=500, resampling=Resampling.bilinear) as vrt:
                a = vrt.read(1).astype("float32")
            dem_marte[name] = a
            d = detectar(a); d["cuerpo"], d["sitio"] = "Marte", name; hm.append(d)
hoyas_ref = pd.concat([hl] + hm, ignore_index=True)

# ---------- Filtro "tipo cráter" y resumen ----------
def tipo_crater(d):
    return (d.diam_km >= 0.6) & (d.diam_km <= 10) & (d.prof_m >= PROF_C) & (d.circ >= 0.6)
hoyas_mx["crater"] = tipo_crater(hoyas_mx); hoyas_ref["crater"] = tipo_crater(hoyas_ref)
res = hoyas_ref[hoyas_ref.crater].groupby(["cuerpo", "sitio"]).agg(n=("diam_km", "size"), diam_km=("diam_km", "median"), prof_m=("prof_m", "median"),
                                                                      prof_diam=("prof_diam", "median"), borde_m=("borde_m", "median"), circ=("circ", "median"))
c_mx = hoyas_mx[hoyas_mx.crater]
res.loc[("Tierra", "México"), :] = [len(c_mx), c_mx.diam_km.median(), c_mx.prof_m.median(), c_mx.prof_diam.median(), c_mx.borde_m.median(), c_mx.circ.median()]
print("\nDepresiones tipo cráter (0.6-10 km, >= 20 m, circulares). Medianas:"); print(res.round(3).to_string())
hoyas_mx.to_csv(f"{OUT}/hoyas_mexico.csv", index=False); hoyas_ref.to_csv(f"{OUT}/hoyas_referencia.csv", index=False)

## 08 · Validación 1: ¿encuentra lo que ya se conoce?

Le damos **11 cráteres mexicanos documentados** (Pinacate, maares de Guanajuato y Puebla, Joya Honda, La Breña) y vemos si hay una detección a menos de 2.5 km. **Las coordenadas de referencia están puestas de memoria y hay que verificarlas.**

**Resultado:** detecta **9 de 11**; 6 pasan el filtro de forma. Los que faltan (Alchichica, La Alberca) tienen **lago**: el relieve ve el espejo de agua, no el fondo.

Esta prueba mide si **encuentra** lo que existe (exhaustividad), **no** cuántas detecciones son falsas (precisión). Eso lo mide la celda 11.

In [ ]:
# 08 · Validación: ¿encuentra los cráteres mexicanos conocidos? ¿dónde están los grupos más densos?
# Coordenadas aproximadas (de memoria, por verificar contra fuentes): sirven para buscar una detección a menos de 2.5 km.
CONOCIDOS = {
    "El Elegante (Pinacate)": (31.846, -113.392), "MacDougal (Pinacate)": (31.974, -113.627), "Sykes (Pinacate)": (31.938, -113.563),
    "Cerro Colorado (Pinacate)": (31.917, -113.302), "Rincón de Parangueo (Gto.)": (20.431, -101.249), "Hoya La Alberca (Gto.)": (20.388, -101.203),
    "Alchichica (Pue.)": (19.414, -97.403), "Atexcac (Pue.)": (19.335, -97.450), "Aljojuca (Pue.)": (19.092, -97.533),
    "Joya Honda (S.L.P.)": (22.418, -100.787), "La Breña (Dgo.)": (24.420, -104.530)}
filas = []
for nombre, (la, lo) in CONOCIDOS.items():
    x, y = to_xy.transform(lo, la)
    dkm = np.hypot(hoyas_mx.x - x, hoyas_mx.y - y) / 1000
    if dkm.min() <= 2.5:
        h = hoyas_mx.loc[dkm.idxmin()]
        filas.append({"cráter": nombre, "detectado": "sí", "a km": round(dkm.min(), 1), "diám. km": round(h.diam_km, 2), "prof. m": round(h.prof_m),
                      "circ": round(h.circ, 2), "borde m": round(h.borde_m), "pasa filtro": "sí" if h.crater else "no"})
    else:
        filas.append({"cráter": nombre, "detectado": "no", "a km": round(dkm.min(), 1)})
val = pd.DataFrame(filas)
print(val.to_string(index=False))
print(f"\nDetectados: {(val.detectado == 'sí').sum()} de {len(val)}; pasan el filtro tipo cráter: {(val.get('pasa filtro') == 'sí').sum()}")

# Densidad: cráteres tipo cráter a menos de 15 km de cada uno
c_mx = hoyas_mx[hoyas_mx.crater].reset_index(drop=True)
from scipy.spatial import cKDTree
tree = cKDTree(c_mx[["x", "y"]].values)
c_mx["vecinos_15km"] = [len(v) - 1 for v in tree.query_ball_point(c_mx[["x", "y"]].values, 15_000)]
xp, yp = to_xy.transform(CONTROL["El Pinacate"][1], CONTROL["El Pinacate"][0])
c_mx["km_pinacate"] = (np.hypot(c_mx.x - xp, c_mx.y - yp) / 1000).round(0)
print(f"\nCráteres tipo cráter en México: {len(c_mx)}. Por franja de latitud:")
print(pd.cut(c_mx.lat, [14, 18, 20, 22, 24, 28, 33]).value_counts().sort_index().to_string())
print(f"A menos de 30 km del centro de El Pinacate: {(c_mx.km_pinacate <= 30).sum()}")
cols = ["lat", "lon", "diam_km", "prof_m", "prof_diam", "circ", "borde_m", "vecinos_15km", "km_pinacate"]
print("\nLos 25 más profundos:"); print(c_mx.nlargest(25, "prof_m")[cols].round(3).to_string(index=False))
print("\nLos 15 con más vecinos (grupos densos):"); print(c_mx.nlargest(15, "vecinos_15km")[cols].round(3).to_string(index=False))
c_mx.to_csv(f"{OUT}/crateres_mexico.csv", index=False)

## 09 · De cráteres a sitios

- **(a) Minas:** marca como mina todo cráter a menos de 2 km de una cantera de OpenStreetMap.
- **(b) Superficie:** NDVI, albedo y cobertura del suelo (MODIS) en cada cráter.
- **(c) Tipo probable:** borde levantado ≥ 10 m → "con borde" (volcánico probable); si no → "sin borde" (dolina o cuenca).
- **(d) Sitios:** agrupa cráteres a menos de 15 km entre sí.
- **(e) Gemelo lunar:** el cráter del polo sur más parecido en diámetro y profundidad, y el **percentil** de su proporción profundidad/diámetro dentro de la población lunar.
- **(f) Conocidos frente a nuevos.**

OpenStreetMap a veces no responde; la celda 10 lo reintenta con otro servidor.

In [ ]:
# 09 · De cráteres a sitios: quitar minas, medir la superficie, agrupar y buscar el gemelo lunar
import requests
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

c = hoyas_mx[hoyas_mx.crater].reset_index(drop=True)

# (a) Minas a cielo abierto: son "cráteres" artificiales. Se marcan con OpenStreetMap (landuse=quarry).
try:
    q = '[out:json][timeout:180];(way["landuse"="quarry"](14,-118.5,33,-86);relation["landuse"="quarry"](14,-118.5,33,-86););out center;'
    js = requests.post("https://overpass-api.de/api/interpreter", data={"data": q}, timeout=240).json()
    qll = np.array([(e["center"]["lon"], e["center"]["lat"]) for e in js["elements"] if "center" in e])
    qx, qy = to_xy.transform(qll[:, 0], qll[:, 1])
    dq, _ = cKDTree(np.c_[qx, qy]).query(c[["x", "y"]].values)
    c["km_mina"] = (dq / 1000).round(1); c["mina"] = c.km_mina <= 2.0
    print(f"Canteras y minas en OpenStreetMap: {len(qll)}. Cráteres a menos de 2 km de una: {int(c.mina.sum())}")
except Exception as e:
    c["km_mina"] = np.nan; c["mina"] = False
    print("No se pudo consultar OpenStreetMap:", str(e)[:150])

# (b) Superficie en cada cráter: vegetación, albedo y cobertura (MODIS, NASA)
try:
    alb = (ee.ImageCollection("MODIS/061/MCD43A3").filterDate("2024-03-01", "2024-06-01").select("Albedo_WSA_shortwave").mean().multiply(0.001).rename("albedo"))
    img = ee.Image.cat([ndvi_col.reduce(ee.Reducer.percentile([90])).multiply(1e-4).rename("ndvi_p90"), alb, lc.rename("lc")])
    pts = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([float(lo), float(la)]), {"k": int(i)}) for i, la, lo in zip(c.index, c.lat, c.lon)])
    props = [f["properties"] for f in img.reduceRegions(pts, ee.Reducer.first(), 500).getInfo()["features"]]
    sup = pd.DataFrame(props).set_index("k")
    for col in ("ndvi_p90", "albedo", "lc"):
        c[col] = sup[col].reindex(c.index).values if col in sup else np.nan
    print("Superficie medida en", int(c.ndvi_p90.notna().sum()), "cráteres")
except Exception as e:
    print("Falló la consulta de superficie:", str(e)[:200])

# (c) Tipo probable por forma: los cráteres volcánicos (maares, conos) tienen borde levantado; las dolinas kársticas no.
c["tipo"] = np.where(c.mina, "mina", np.where(c.borde_m >= 10, "con borde (volcánico probable)", "sin borde (dolina o cuenca probable)"))
print("\nTipos:", c.tipo.value_counts().to_dict())

# (d) Sitios: grupos de cráteres a menos de 15 km entre sí
pairs = cKDTree(c[["x", "y"]].values).query_pairs(15_000, output_type="ndarray")
g = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(len(c), len(c)))
c["sitio_id"] = connected_components(g, directed=False)[1]

# (e) Gemelo lunar: el cráter del polo sur más parecido en diámetro y profundidad (escala logarítmica)
L = hoyas_ref[(hoyas_ref.cuerpo == "Luna") & hoyas_ref.crater].reset_index(drop=True)
tl = cKDTree(np.c_[np.log(L.diam_km), np.log(L.prof_m)])
dg, ig = tl.query(np.c_[np.log(c.diam_km), np.log(c.prof_m)])
c["gem_lat"], c["gem_lon"], c["gem_diam_km"], c["gem_prof_m"] = L.lat.values[ig], L.lon.values[ig], L.diam_km.values[ig], L.prof_m.values[ig]
# qué tan "lunar" es la proporción profundidad/diámetro: percentil dentro de la población lunar
c["pct_lunar_prof_diam"] = np.searchsorted(np.sort(L.prof_diam.values), c.prof_diam.values) / len(L) * 100

# (f) Conocidos frente a nuevos
kx, ky = to_xy.transform([v[1] for v in CONOCIDOS.values()], [v[0] for v in CONOCIDOS.values()])
c["km_conocido"] = (cKDTree(np.c_[kx, ky]).query(c[["x", "y"]].values)[0] / 1000).round(1)
xp, yp = to_xy.transform(CONTROL["El Pinacate"][1], CONTROL["El Pinacate"][0])
c["km_pinacate"] = (np.hypot(c.x - xp, c.y - yp) / 1000).round(0)
c["conocido"] = (c.km_conocido <= 5) | (c.km_pinacate <= 40)

vol = c[c.tipo.str.startswith("con borde")]
sit = vol.groupby("sitio_id").agg(n=("diam_km", "size"), lat=("lat", "mean"), lon=("lon", "mean"), diam_max=("diam_km", "max"), prof_max=("prof_m", "max"),
                                  borde=("borde_m", "median"), ndvi=("ndvi_p90", "median"), albedo=("albedo", "median"), conocido=("conocido", "any")).reset_index()
print(f"\nCráteres con borde levantado, sin minas: {len(vol)} en {len(sit)} sitios. Conocidos: {int(sit.conocido.sum())} sitios")
print("\nSitios con borde levantado, ordenados por profundidad máxima (los 30 primeros):")
print(sit.sort_values("prof_max", ascending=False).head(30).round(3).to_string(index=False))
print("\nCráteres individuales con borde, poca vegetación (NDVI < 0.35) y no conocidos:")
arid = vol[(vol.ndvi_p90 < 0.35) & ~vol.conocido].sort_values("prof_m", ascending=False)
print(arid[["lat", "lon", "diam_km", "prof_m", "prof_diam", "circ", "borde_m", "ndvi_p90", "albedo", "km_mina", "pct_lunar_prof_diam"]].head(30).round(3).to_string(index=False))
c.to_csv(f"{OUT}/crateres_mexico.csv", index=False); sit.to_csv(f"{OUT}/sitios_mexico.csv", index=False)

## 10 · Exportar para la página web

Genera `web_data.zip` con:

- `fichas.json`: 87 cráteres (con nombre, con borde o minas), cada uno con su gemelo lunar.
- `crateres.json`: los 668 para el mapa.
- `juego.json` + imágenes: 66 rondas (28 Tierra, 24 Luna, 14 Marte).
- `referencia.json`: poblaciones de cráteres de la Luna y Marte, tablas y validación.
- `marte_llanuras.json`: llanuras mexicanas parecidas a Oxia Planum.
- Imágenes de relieve sombreado: **misma luz y mismo tratamiento** para los tres mundos.

Los **nombres de las minas** (Peñasquito, Real de Ángeles, La Herradura, Hércules, Cananea) se asignan por coordenadas y **están por verificar**.

El contenido va a `web/public/data/` en el repositorio.

In [ ]:
# 10 · Exportar los datos para la página web: cráteres, gemelos lunares, imágenes de relieve y juego
import json, shutil
from PIL import Image
WEB = "/content/web_data"; shutil.rmtree(WEB, ignore_errors=True); os.makedirs(f"{WEB}/img", exist_ok=True)
P = 32   # medio lado del recorte: 64 px = 12.8 km

# --- Minas: OpenStreetMap (si responde) + identificación por coordenadas (por verificar) ---
MINAS = {"Peñasquito (Zac.)": (24.649, -101.673), "Real de Ángeles (Zac.)": (22.414, -101.911), "La Herradura (Son.)": (31.147, -112.863),
         "Hércules (Coah.)": (28.046, -103.782), "Cananea (Son.)": (30.953, -110.320)}
osm_ok = False
for url in ("https://overpass.kumi.systems/api/interpreter", "https://overpass-api.de/api/interpreter"):
    try:
        q = '[out:json][timeout:180];(way["landuse"="quarry"](14,-118.5,33,-86);relation["landuse"="quarry"](14,-118.5,33,-86););out center;'
        r = requests.post(url, data={"data": q}, headers={"User-Agent": "analog-research-notebook/1.0"}, timeout=240)
        qll = np.array([(e["center"]["lon"], e["center"]["lat"]) for e in r.json()["elements"] if "center" in e])
        qx, qy = to_xy.transform(qll[:, 0], qll[:, 1])
        c["km_mina"] = (cKDTree(np.c_[qx, qy]).query(c[["x", "y"]].values)[0] / 1000).round(1)
        osm_ok = True; print(f"OpenStreetMap: {len(qll)} canteras/minas; cráteres a menos de 2 km: {int((c.km_mina <= 2).sum())}"); break
    except Exception as e:
        print("OSM falló en", url.split('/')[2], "->", str(e)[:80])
mx_, my_ = to_xy.transform([v[1] for v in MINAS.values()], [v[0] for v in MINAS.values()])
dman, iman = cKDTree(np.c_[mx_, my_]).query(c[["x", "y"]].values)
c["nombre"] = ""
c.loc[dman <= 2500, "nombre"] = np.array(list(MINAS))[iman[dman <= 2500]]
c["mina"] = (dman <= 2500) | ((c.km_mina <= 2.0) if osm_ok else False)
dk, ik = cKDTree(np.c_[kx, ky]).query(c[["x", "y"]].values)
c.loc[dk <= 2500, "nombre"] = np.array(list(CONOCIDOS))[ik[dk <= 2500]]
c["tipo"] = np.where(c.mina, "mina", np.where(c.borde_m >= 10, "con_borde", "sin_borde"))
c["cid"] = ["MX%04d" % i for i in range(len(c))]
print("Tipos:", c.tipo.value_counts().to_dict(), "| con nombre:", int((c.nombre != "").sum()))

# --- Imágenes: relieve sombreado en gris, misma luz y mismo tratamiento para los tres cuerpos ---
def sombreado(dem, az=315.0, alt=35.0):
    gy, gx = np.gradient(dem.astype("float64"), BASE)
    slope = np.pi / 2 - np.arctan(np.hypot(gx, gy)); aspect = np.arctan2(-gx, gy)
    a, h = np.radians(360 - az + 90), np.radians(alt)
    sh = np.sin(h) * np.sin(slope) + np.cos(h) * np.cos(slope) * np.cos(a - aspect)
    lo, hi = np.percentile(sh, [2, 98])
    return np.clip((sh - lo) / max(hi - lo, 1e-6), 0, 1)
def guardar(dem, nombre):
    Image.fromarray((sombreado(dem) * 255).astype("uint8")).resize((320, 320), Image.LANCZOS).save(f"{WEB}/img/{nombre}.png", optimize=True)
def recorte(arr, i, j):
    i, j = int(round(i)), int(round(j))
    if i - P < 0 or j - P < 0 or i + P > arr.shape[0] or j + P > arr.shape[1]:
        return None
    return arr[i - P:i + P, j - P:j + P]
def recorte_mx(r):
    arr, xl, yt = dem_bloques[(int(r.br), int(r.bc))]
    return recorte(arr, r.i, r.j)

# --- Fichas: cráteres con nombre + los de borde levantado + minas; cada uno con su gemelo lunar ---
sel = c[(c.nombre != "") | (c.tipo == "con_borde") | c.mina].copy()
fichas = []
for r in sel.itertuples():
    p = recorte_mx(r)
    g = L.iloc[tl.query([[np.log(r.diam_km), np.log(r.prof_m)]])[1][0]]
    pg = recorte(moon200, g.i, g.j)
    if p is None or pg is None:
        continue
    guardar(p, r.cid); guardar(pg, r.cid + "_luna")
    fichas.append({"id": r.cid, "nombre": r.nombre, "tipo": r.tipo, "lat": round(r.lat, 4), "lon": round(r.lon, 4), "diam_km": round(r.diam_km, 2),
                   "prof_m": round(r.prof_m), "prof_diam": round(r.prof_diam, 3), "circ": round(r.circ, 2), "borde_m": round(r.borde_m),
                   "ndvi": None if pd.isna(r.ndvi_p90) else round(r.ndvi_p90, 2), "albedo": None if pd.isna(r.albedo) else round(r.albedo, 3),
                   "pct_lunar": round(r.pct_lunar_prof_diam), "conocido": bool(r.conocido), "km_pinacate": int(r.km_pinacate),
                   "gemelo": {"lat": round(float(g.lat), 3), "lon": round(float(g.lon), 3), "diam_km": round(float(g.diam_km), 2), "prof_m": round(float(g.prof_m))}})
json.dump(fichas, open(f"{WEB}/fichas.json", "w"), ensure_ascii=False)

# --- Mapa: todos los cráteres detectados, ligeros ---
todos = [{"id": r.cid, "lat": round(r.lat, 4), "lon": round(r.lon, 4), "d": round(r.diam_km, 2), "p": round(r.prof_m), "t": r.tipo, "n": r.nombre} for r in c.itertuples()]
json.dump(todos, open(f"{WEB}/crateres.json", "w"), ensure_ascii=False)

# --- Referencia: poblaciones de cráteres en la Luna y Marte, y resumen por cuadros ---
R = hoyas_ref[hoyas_ref.crater]
refj = {"luna_polo": R[R.sitio == "Polo sur (85-90°S)"][["diam_km", "prof_m"]].round(2).values.tolist(),
        "marte": {s: R[R.sitio == s][["diam_km", "prof_m"]].round(2).values.tolist() for s in SITIOS_MARTE},
        "mexico": c[["diam_km", "prof_m"]].round(2).values.tolist(),
        "resumen": json.loads(res.round(3).reset_index().to_json(orient="records")),
        "terreno": json.loads(tabla.round(2).reset_index().to_json(orient="records")),
        "validacion": json.loads(val.to_json(orient="records"))}
json.dump(refj, open(f"{WEB}/referencia.json", "w"), ensure_ascii=False)

# --- Marte: llanuras mexicanas frente a Oxia Planum (puntaje por cuadros) ---
ll = cand.nsmallest(25, "f_oxia")[["cell_id", "lat", "lon", "slope_p50", "slope_p90", "relieve", "dep_n", "ndvi_p90", "f_desnudo", "f_oxia"]].round(3)
json.dump(json.loads(ll.to_json(orient="records")), open(f"{WEB}/marte_llanuras.json", "w"), ensure_ascii=False)

# --- Juego: recortes de cráteres de los tres cuerpos ---
rng = np.random.default_rng(7); juego = []
def add(dem, cuerpo, k, **info):
    if dem is None: return
    guardar(dem, f"j_{cuerpo}_{k}"); juego.append(dict(img=f"j_{cuerpo}_{k}.png", cuerpo=cuerpo, **info))
lsel = L[(L.sitio == "Polo sur (85-90°S)") & L.diam_km.between(0.8, 3) & (L.prof_m >= 40)]
for k, r in enumerate(lsel.sample(min(24, len(lsel)), random_state=7).itertuples()):
    add(recorte(moon200, r.i, r.j), "luna", k, lugar="Polo sur lunar", lat=round(r.lat, 2), lon=round(r.lon, 2), diam_km=round(r.diam_km, 2), prof_m=round(r.prof_m))
for s in SITIOS_MARTE:
    ms = R[(R.sitio == s) & R.diam_km.between(0.8, 3)]
    for k, r in enumerate(ms.sample(min(8, len(ms)), random_state=7).itertuples()):
        add(recorte(dem_marte[s], r.i, r.j), "marte", f"{s[:3]}{k}", lugar=s, diam_km=round(r.diam_km, 2), prof_m=round(r.prof_m))
tierra = pd.concat([c[c.nombre != ""], c[(c.tipo == "sin_borde") & (c.nombre == "")].nlargest(10, "prof_m"), c[(c.tipo == "con_borde") & (c.nombre == "")].nlargest(6, "prof_m")]).drop_duplicates("cid")
for r in tierra.itertuples():
    add(recorte_mx(r), "tierra", r.cid, lugar=r.nombre or "México", tipo=r.tipo, lat=round(r.lat, 3), lon=round(r.lon, 3), diam_km=round(r.diam_km, 2), prof_m=round(r.prof_m))
json.dump(juego, open(f"{WEB}/juego.json", "w"), ensure_ascii=False)

# --- Tablas y código, y zip final ---
c.to_csv(f"{OUT}/crateres_mexico.csv", index=False)
for f in ("crateres_mexico.csv", "hoyas_referencia.csv", "candidatos_mexico_10km.csv", "referencias_luna_marte_10km.csv"):
    if os.path.exists(f"{OUT}/{f}"): shutil.copy(f"{OUT}/{f}", f"{WEB}/{f}")
get_ipython().run_line_magic("history", f"-f {WEB}/codigo_sesion.py")
z = shutil.make_archive("/content/web_data", "zip", WEB)
print(f"Fichas: {len(fichas)} | cráteres en el mapa: {len(todos)} | rondas de juego: {pd.Series([j['cuerpo'] for j in juego]).value_counts().to_dict()}")
print(f"Zip: {z} ({os.path.getsize(z)/1e6:.1f} MB), {len(os.listdir(WEB + '/img'))} imágenes")
print(pd.DataFrame(fichas)[["id", "nombre", "tipo", "lat", "lon", "diam_km", "prof_m", "prof_diam", "borde_m", "ndvi", "pct_lunar"]][lambda d: d.nombre != ""].to_string(index=False))

## 11 · Validación 2: ¿qué tan seguido acierta?

**Dato extra:** Sentinel-2 (imagen de satélite en color real, mediana 2023–2025), solo para **revisar a ojo**.

Hace pares de imágenes (relieve a la izquierda, satélite a la derecha, 12.8 km de lado, con un círculo del tamaño de la detección) para:

1. los 11 cráteres conocidos,
2. una **muestra al azar** (semilla 42) de 16 "con borde", 16 "sin borde" y 8 "minas",
3. 12 detecciones del polo sur lunar.

**Resultado (lectura a ojo, por confirmar):** 1 de 16 "con borde", 0 de 16 "sin borde" y 4 de 8 "minas" parecen reales. Los aciertos son grandes, profundos y en desierto; los fallos, pequeños, poco profundos y en sierras con selva.

Las imágenes alimentan `informe/validacion_detector.pdf`.

In [ ]:
# 11 · Imágenes para el informe de validación: relieve + satélite de cráteres conocidos y de una muestra al azar
import json, shutil, io, requests
from PIL import Image, ImageDraw
from concurrent.futures import ThreadPoolExecutor
INF = "/content/informe"; shutil.rmtree(INF, ignore_errors=True); os.makedirs(f"{INF}/img", exist_ok=True)
PX = 320; KM = 2 * P * BASE / 1000          # 12.8 km por recorte

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterDate("2023-01-01", "2025-12-31")
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10)).select(["B4", "B3", "B2"]).median()
      .visualize(min=0, max=2500, gamma=1.2))

def bloque_de(x, y):
    for (r_, c_), (arr, xl, yt) in dem_bloques.items():
        i, j = (yt - y) / BASE - 0.5, (x - xl) / BASE - 0.5
        if P <= i < arr.shape[0] - P and P <= j < arr.shape[1] - P:
            return arr, i, j
    return None, None, None

def circulo(im, diam_km, color):
    d = ImageDraw.Draw(im); r = max(diam_km / KM * PX / 2, 4); c = PX / 2
    d.ellipse([c - r, c - r, c + r, c + r], outline=color, width=3); return im

def relieve_png(lat, lon, diam_km, nombre):
    x, y = to_xy.transform(lon, lat); arr, i, j = bloque_de(x, y)
    if arr is None: return False
    im = Image.fromarray((sombreado(recorte(arr, i, j)) * 255).astype("uint8")).resize((PX, PX), Image.LANCZOS).convert("RGB")
    if diam_km: circulo(im, diam_km, (230, 60, 40))
    im.save(f"{INF}/img/{nombre}_rel.png", optimize=True); return True

def satelite_png(lat, lon, diam_km, nombre):
    x, y = to_xy.transform(lon, lat); h = KM * 500
    reg = ee.Geometry.Rectangle([x - h, y - h, x + h, y + h], ee.Projection(CRS), False)
    for k in range(3):
        try:
            url = s2.getThumbURL({"region": reg, "dimensions": PX, "crs": CRS, "format": "png"})
            im = Image.open(io.BytesIO(requests.get(url, timeout=60).content)).convert("RGB").resize((PX, PX))
            if diam_km: circulo(im, diam_km, (255, 220, 0))
            im.save(f"{INF}/img/{nombre}_sat.png", optimize=True); return True
        except Exception as e:
            err = str(e)[:120]; time.sleep(2)
    print("satélite falló", nombre, err); return False

items = []
# (1) Cráteres conocidos: se dibuja el lugar del catálogo; el círculo es la detección más cercana si la hubo
for k, (nombre, (la, lo)) in enumerate(CONOCIDOS.items()):
    x, y = to_xy.transform(lo, la); dkm = np.hypot(hoyas_mx.x - x, hoyas_mx.y - y) / 1000
    h = hoyas_mx.loc[dkm.idxmin()] if dkm.min() <= 2.5 else None
    items.append({"grupo": "conocido", "id": f"K{k:02d}", "nombre": nombre, "lat": la, "lon": lo,
                  "detectado": h is not None, "pasa_filtro": bool(h.crater) if h is not None else False,
                  "diam_km": round(float(h.diam_km), 2) if h is not None else None, "prof_m": round(float(h.prof_m)) if h is not None else None,
                  "circ": round(float(h.circ), 2) if h is not None else None, "borde_m": round(float(h.borde_m)) if h is not None else None,
                  "c_lat": round(float(h.lat), 4) if h is not None else la, "c_lon": round(float(h.lon), 4) if h is not None else lo})
# (2) Muestra al azar por tipo (semilla fija para que sea reproducible)
for tipo, n in (("con_borde", 16), ("sin_borde", 16), ("mina", 8)):
    sub = c[c.tipo == tipo]
    for r in sub.sample(min(n, len(sub)), random_state=42).itertuples():
        items.append({"grupo": tipo, "id": r.cid, "nombre": r.nombre, "lat": round(r.lat, 4), "lon": round(r.lon, 4), "c_lat": round(r.lat, 4), "c_lon": round(r.lon, 4),
                      "diam_km": round(r.diam_km, 2), "prof_m": round(r.prof_m), "prof_diam": round(r.prof_diam, 3), "circ": round(r.circ, 2),
                      "borde_m": round(r.borde_m), "ndvi": None if pd.isna(r.ndvi_p90) else round(r.ndvi_p90, 2), "pct_lunar": round(r.pct_lunar_prof_diam)})

def hacer(it):
    ok1 = relieve_png(it["c_lat"], it["c_lon"], it.get("diam_km"), it["id"])
    ok2 = satelite_png(it["c_lat"], it["c_lon"], it.get("diam_km"), it["id"])
    it["img_rel"], it["img_sat"] = ok1, ok2; return it
t0 = time.time()
with ThreadPoolExecutor(6) as ex: items = list(ex.map(hacer, items))

# (3) Muestra de detecciones en la Luna (polo sur), solo relieve
luna_items = []
Lp = L[L.sitio == "Polo sur (85-90°S)"]
for k, r in enumerate(Lp.sample(12, random_state=42).itertuples()):
    p_ = recorte(moon200, r.i, r.j)
    if p_ is None: continue
    im = Image.fromarray((sombreado(p_) * 255).astype("uint8")).resize((PX, PX), Image.LANCZOS).convert("RGB")
    circulo(im, r.diam_km, (230, 60, 40)).save(f"{INF}/img/L{k:02d}_rel.png", optimize=True)
    luna_items.append({"id": f"L{k:02d}", "lat": round(r.lat, 2), "lon": round(r.lon, 2), "diam_km": round(r.diam_km, 2), "prof_m": round(r.prof_m), "prof_diam": round(r.prof_diam, 3)})

# (4) Números de resumen
resumen = {"n_mexico": int(len(c)), "tipos": c.tipo.value_counts().to_dict(), "n_hoyas_total": int(len(hoyas_mx)),
           "n_luna_polo": int((hoyas_ref.crater & (hoyas_ref.sitio == "Polo sur (85-90°S)")).sum()),
           "lat_bandas": {str(k): int(v) for k, v in pd.cut(c.lat, [14, 18, 20, 22, 24, 28, 33]).value_counts().sort_index().items()},
           "medianas": json.loads(res.round(3).reset_index().to_json(orient="records")),
           "candidatos": int(len(cand)), "cuadros_mx": int(len(mx))}
json.dump({"items": items, "luna": luna_items, "resumen": resumen}, open(f"{INF}/informe.json", "w"), ensure_ascii=False)
z = shutil.make_archive("/content/informe", "zip", INF)
print(f"{len(items)} sitios ({sum(i['img_sat'] for i in items)} con satélite), {len(luna_items)} lunares. {time.time()-t0:.0f} s. Zip {os.path.getsize(z)/1e6:.1f} MB")

## 12 · Todas las detecciones en zonas áridas

Las mismas imágenes para **las 47 detecciones con NDVI < 0.35**: la lista que proponemos revisar a mano y dejar en la página.

Lectura preliminar: las más profundas (≳ 60 m) son reales (tajos mineros y cráteres del Pinacate y Guanajuato); debajo aparecen hondonadas entre dunas, presas y lagos (depresiones reales, pero no cráteres).

In [ ]:
# 12 · Imágenes de TODAS las detecciones en zonas áridas (NDVI p90 < 0.35): la lista que de verdad importa
arid_items = []
for r in c[c.ndvi_p90 < 0.35].sort_values("prof_m", ascending=False).itertuples():
    arid_items.append({"grupo": "arida", "id": "A_" + r.cid, "cid": r.cid, "nombre": r.nombre, "tipo": r.tipo, "lat": round(r.lat, 4), "lon": round(r.lon, 4),
                       "c_lat": round(r.lat, 4), "c_lon": round(r.lon, 4), "diam_km": round(r.diam_km, 2), "prof_m": round(r.prof_m),
                       "prof_diam": round(r.prof_diam, 3), "circ": round(r.circ, 2), "borde_m": round(r.borde_m), "ndvi": round(r.ndvi_p90, 2),
                       "pct_lunar": round(r.pct_lunar_prof_diam), "km_mina": None if pd.isna(r.km_mina) else float(r.km_mina)})
t0 = time.time()
with ThreadPoolExecutor(6) as ex: arid_items = list(ex.map(hacer, arid_items))
d = json.load(open(f"{INF}/informe.json")); d["aridas"] = arid_items
d["ndvi_bandas"] = {str(k): int(v) for k, v in pd.cut(c.ndvi_p90, [0, .2, .35, .5, .7, 1]).value_counts().sort_index().items()}
json.dump(d, open(f"{INF}/informe.json", "w"), ensure_ascii=False)
z = shutil.make_archive("/content/informe", "zip", INF)
print(f"{len(arid_items)} detecciones áridas con imagen. {time.time()-t0:.0f} s. Zip {os.path.getsize(z)/1e6:.1f} MB")

## 13 · Experimento: ¿un relieve sin árboles quita los falsos?

**Hipótesis:** Copernicus GLO-30 mide copas de árboles; los huecos del dosel parecerían hoyos. **FABDEM** es el mismo Copernicus con bosques y edificios quitados.

**Prueba:** repetir el detector con FABDEM en 4 bloques de selva (los de más detecciones) y 2 áridos (Pinacate y Zacatecas).

**Resultado:** en selva bajó de 181 a 153 detecciones (**solo −15 %**); en zonas áridas conservó los 4 cráteres reales. **Hipótesis descartada:** los falsos son depresiones **reales** del terreno (dolinas kársticas, cañadas cerradas) que a 200 m cumplen las reglas de forma. La solución es filtrar por zona árida, no cambiar de relieve.

In [ ]:
# 13 · Experimento: ¿FABDEM (Copernicus sin bosques ni edificios) elimina los falsos cráteres de selva?
# Se re-detecta con FABDEM en los mismos bloques de 160 km de una muestra de detecciones y se compara con GLO-30.
fab = ee.ImageCollection("projects/sat-io/open-datasets/FABDEM")
print("FABDEM imágenes:", fab.size().getInfo(), "| bandas:", fab.first().bandNames().getInfo())
fab_ee = (fab.mosaic().setDefaultProjection(fab.first().projection())
          .reduceResolution(ee.Reducer.mean(), maxPixels=256).unmask(0).toFloat().rename("dem"))

def detectar_bloque(img, rc):
    r, c_ = rc; h, w = min(BLOCK, ny - r), min(BLOCK, nx - c_)
    xl, yt = x0 + c_ * CELL - M * BASE, y1 - r * CELL + M * BASE
    a = ee_pixels(img, xl, yt, w * N + 2 * M, h * N + 2 * M, BASE)["dem"]
    d = detectar(a)
    if d.empty: return d
    d = d[(d.i >= M) & (d.i < M + h * N) & (d.j >= M) & (d.j < M + w * N)].copy()
    d["x"], d["y"] = xl + (d.j + 0.5) * BASE, yt - (d.i + 0.5) * BASE
    return d[tipo_crater(d)]

# Bloques: 4 de selva (Chiapas/Oaxaca/Huasteca) con muchas detecciones + 2 áridos con cráteres reales (Pinacate, Zacatecas)
cnt = c.assign(rc=list(zip(c.br.astype(int), c.bc.astype(int))))
selva = cnt[cnt.ndvi_p90 > 0.7].rc.value_counts().head(4).index.tolist()
arido = [tuple(cnt.loc[cnt.cid == k, "rc"].iloc[0]) for k in ("MX0008", "MX0207")]
filas = []
for rc in selva + arido:
    g = detectar_bloque(dem_ee, rc); f = detectar_bloque(fab_ee, rc)
    # ¿siguen las detecciones reales? (cráteres áridos conocidos del bloque)
    reales = c[(c.br == rc[0]) & (c.bc == rc[1]) & (c.ndvi_p90 < 0.35) & (c.prof_m >= 60)]
    keep = sum(int((np.hypot(f.x - r_.x, f.y - r_.y) < 1500).any()) for r_ in reales.itertuples()) if len(f) else 0
    filas.append({"bloque": rc, "zona": "selva" if rc in selva else "árida", "GLO-30": len(g), "FABDEM": len(f),
                  "reales (prof≥60 m, árido)": len(reales), "reales que FABDEM conserva": keep})
print(pd.DataFrame(filas).to_string(index=False))

## 99 · Descargar resultados

Baja al equipo:
- `web_data.zip` → descomprimir en `web/public/data/`.
- `informe.zip` → descomprimir en `informe/datos/` y correr `informe/generar.py` para el PDF.

Colab pide permiso del navegador para descargar; si no baja, ejecuta la celda tú con el botón de play.

In [ ]:
# 99 · Descargar al equipo los resultados (si no baja, ejecuta esta celda tú con el botón de play)
from google.colab import files
for z in ("/content/web_data.zip", "/content/informe.zip"):
    if os.path.exists(z):
        files.download(z)
